In [32]:
from typing import Dict, List, Set, Tuple, Optional
from collections import defaultdict
import random
from nltk import Tree
from nltk.draw.util import CanvasFrame
from nltk.draw import TreeWidget
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

class CFGParser:
    def __init__(self):
        self.rules = {
            'S': [['NP', 'VP']],
            'NP': [['Det', 'N'], ['Det', 'Adj', 'N'], ['NP', 'PP'], ['NP', 'RelClause']],
            'VP': [['V', 'NP'], ['V'], ['VP', 'PP']],
            'PP': [['Prep', 'NP']],
            'RelClause': [['RelPro', 'VP']],
            'Det': [['the'], ['a']],
            'N': [['boy'], ['girl'], ['dog'], ['telescope'], ['man'], ['park']],
            'Adj': [['tall'], ['young'], ['old']],
            'RelPro': [['who'], ['that']],
            'V': [['saw'], ['chased'], ['liked'], ['ate']],
            'Prep': [['with'], ['in'], ['near']]
        }
        self.terminals = {'Det', 'N', 'Adj', 'RelPro', 'V', 'Prep'}
        self.non_terminals = set(self.rules.keys())
    
    def display_rules(self):
        for lhs, rhs_list in self.rules.items():
            for rhs in rhs_list:
                rhs_str = ' '.join(rhs)
                print(f"{lhs:15} → {rhs_str}")

parser = CFGParser()
parser.display_rules()


S               → NP VP
NP              → Det N
NP              → Det Adj N
NP              → NP PP
NP              → NP RelClause
VP              → V NP
VP              → V
VP              → VP PP
PP              → Prep NP
RelClause       → RelPro VP
Det             → the
Det             → a
N               → boy
N               → girl
N               → dog
N               → telescope
N               → man
N               → park
Adj             → tall
Adj             → young
Adj             → old
RelPro          → who
RelPro          → that
V               → saw
V               → chased
V               → liked
V               → ate
Prep            → with
Prep            → in
Prep            → near


In [35]:
class SentenceGenerator:
    def __init__(self, cfg_parser):
        self.cfg_parser = cfg_parser
        self.generated = set()
    
    def generate_from_rule(self, symbol, max_depth=5, force_feature=None):
        if max_depth == 0:
            return None
        
        if symbol not in self.cfg_parser.rules:
            return None
        
        productions = self.cfg_parser.rules[symbol]
        
        if force_feature == 'rel_clause' and symbol == 'NP':
            rel_clause_prod = [p for p in productions if 'RelClause' in p]
            if rel_clause_prod:
                production = random.choice(rel_clause_prod)
            else:
                production = random.choice(productions)
        elif force_feature == 'pp' and symbol == 'NP':
            pp_prod = [p for p in productions if 'PP' in p]
            if pp_prod:
                production = random.choice(pp_prod)
            else:
                production = random.choice(productions)
        else:
            production = random.choice(productions)
        
        sentence_parts = []
        for item in production:
            if item in self.cfg_parser.rules:
                part = self.generate_from_rule(item, max_depth - 1, force_feature)
                if part is None:
                    return None
                sentence_parts.append(part)
            else:
                sentence_parts.append(item)
        
        return ' '.join(sentence_parts)
    
    def generate_sentences(self, count=10):
        sentences = []
        attempts = 0
        max_attempts = 2000
        rel_count = 0
        pp_count = 0
        
        while len(sentences) < count and attempts < max_attempts:
            if rel_count < 2 and attempts % 3 == 0:
                sent = self.generate_from_rule('S', force_feature='rel_clause')
            elif pp_count < 2 and attempts % 3 == 1:
                sent = self.generate_from_rule('S', force_feature='pp')
            else:
                sent = self.generate_from_rule('S')
            
            if sent and sent not in self.generated:
                self.generated.add(sent)
                tokens = sent.split()
                
                if self.has_relative_clause(tokens):
                    rel_count += 1
                if self.has_pp(tokens):
                    pp_count += 1
                
                sentences.append(sent)
            
            attempts += 1
        
        return sentences[:count]
    
    def has_relative_clause(self, tokens):
        rel_pros = {'who', 'that'}
        return any(token in rel_pros for token in tokens)
    
    def has_pp(self, tokens):
        preps = {'with', 'in', 'near'}
        return any(token in preps for token in tokens)

generator = SentenceGenerator(parser)

generated_sentences = []
while len(generated_sentences) < 10:
    new_sents = generator.generate_sentences(10)
    generated_sentences.extend(new_sents)

generated_sentences = generated_sentences[:10]

rel_clause_count = 0
pp_count = 0

for idx, sent in enumerate(generated_sentences, 1):
    tokens = sent.split()
    has_rel = generator.has_relative_clause(tokens)
    has_pp = generator.has_pp(tokens)
    
    rel_clause_count += has_rel
    pp_count += has_pp
    
    markers = []
    if has_rel:
        markers.append("[REL]")
    if has_pp:
        markers.append("[PP]")
    
    marker_str = " ".join(markers) if markers else ""
    print(f"{idx:2}. {sent:50} {marker_str}")

print(f"\nRelative Clauses: {rel_clause_count}, Prepositional Phrases: {pp_count}")


 1. the man chased a old telescope near a tall man     [PP]
 2. the tall man ate the park                          
 3. a tall man liked                                   
 4. a young telescope chased the young man             
 5. the tall park saw a young boy                      
 6. the tall man chased a young dog                    
 7. the telescope ate                                  
 8. the park that liked chased                         [REL]
 9. a park chased the old girl                         
10. a tall girl chased                                 

Relative Clauses: 1, Prepositional Phrases: 1


In [37]:
class ParseTree:
    def __init__(self, symbol, children=None):
        self.symbol = symbol
        self.children = children if children else []
    
    def is_terminal(self):
        return len(self.children) == 0
    
    def to_string(self, indent=0):
        prefix = "  " * indent
        result = f"{prefix}{self.symbol}"
        for child in self.children:
            result += "\n" + child.to_string(indent + 1)
        return result
    
    def to_nltk_tree(self):
        if self.is_terminal():
            return Tree(self.symbol, [self.symbol])
        else:
            children = [child.to_nltk_tree() for child in self.children]
            return Tree(self.symbol, children)

class TopDownParser:
    def __init__(self, cfg_parser):
        self.cfg_parser = cfg_parser
        self.derivation_steps = []
        self.memo = {}
        self.recursion_limit = 50
    
    def parse(self, tokens):
        self.derivation_steps = []
        self.memo = {}
        self.derivation_steps.append(f"Input: {' '.join(tokens)}")
        self.derivation_steps.append(f"Start: S")
        
        tree, pos, depth = self.parse_symbol('S', tokens, 0, 0)
        
        if tree and pos == len(tokens):
            return tree, self.derivation_steps
        else:
            return None, self.derivation_steps
    
    def parse_symbol(self, symbol, tokens, pos, depth):
        if pos > len(tokens) or depth > self.recursion_limit:
            return None, pos, depth
        
        if symbol not in self.cfg_parser.rules:
            if pos < len(tokens) and tokens[pos] == symbol:
                self.derivation_steps.append(f"Match terminal '{symbol}' at position {pos}")
                return ParseTree(symbol), pos + 1, depth
            else:
                return None, pos, depth
        
        for production in self.cfg_parser.rules[symbol]:
            children = []
            current_pos = pos
            valid = True
            
            for item in production:
                tree, current_pos, depth = self.parse_symbol(item, tokens, current_pos, depth + 1)
                if tree is None:
                    valid = False
                    break
                children.append(tree)
            
            if valid:
                self.derivation_steps.append(f"Reduce: {symbol} → {' '.join(production)}")
                return ParseTree(symbol, children), current_pos, depth
        
        return None, pos, depth


test_sentences = generated_sentences[:3]

for test_idx, test_sent in enumerate(test_sentences, 1):
    print(f"\n--- Test Sentence {test_idx} ---")
    print(f"Sentence: {test_sent}")
    
    tokens = test_sent.split()
    
    parser_td = TopDownParser(parser)
    tree_1, derivation = parser_td.parse(tokens)
    
    if tree_1:
        print(f"Status: ACCEPTED")
        print(f"\nDerivation Steps:")
        for step in derivation:
            print(f"  {step}")
        print(f"\nParse Tree:")
        print(tree_1.to_string())
    else:
        print(f"Status: REJECTED")
        print(f"\nDerivation Steps:")
        for step in derivation:
            print(f"  {step}")



--- Test Sentence 1 ---
Sentence: the man chased a old telescope near a tall man
Status: REJECTED

Derivation Steps:
  Input: the man chased a old telescope near a tall man
  Start: S
  Match terminal 'the' at position 0
  Reduce: Det → the
  Match terminal 'man' at position 1
  Reduce: N → man
  Reduce: NP → Det N
  Match terminal 'chased' at position 2
  Reduce: V → chased
  Match terminal 'a' at position 3
  Reduce: Det → a
  Match terminal 'a' at position 3
  Reduce: Det → a
  Match terminal 'old' at position 4
  Reduce: Adj → old
  Match terminal 'telescope' at position 5
  Reduce: N → telescope
  Reduce: NP → Det Adj N
  Reduce: VP → V NP
  Reduce: S → NP VP

--- Test Sentence 2 ---
Sentence: the tall man ate the park
Status: ACCEPTED

Derivation Steps:
  Input: the tall man ate the park
  Start: S
  Match terminal 'the' at position 0
  Reduce: Det → the
  Match terminal 'the' at position 0
  Reduce: Det → the
  Match terminal 'tall' at position 1
  Reduce: Adj → tall
  Match te

In [39]:
class BottomUpParser:
    def __init__(self, cfg_parser):
        self.cfg_parser = cfg_parser
        self.steps = []
        self.build_reverse_rules()
    
    def build_reverse_rules(self):
        self.reverse_rules = defaultdict(list)
        for lhs, productions in self.cfg_parser.rules.items():
            for production in productions:
                key = tuple(production)
                self.reverse_rules[key].append(lhs)
    
    def parse(self, tokens):
        self.steps = []
        stack = []
        input_queue = list(tokens)
        
        self.steps.append(f"Input: {' '.join(tokens)}")
        self.steps.append("Starting shift-reduce parsing...\n")
        
        while input_queue or len(stack) > 1:
            if not input_queue:
                if len(stack) == 1 and stack[0] == 'S':
                    self.steps.append("ACCEPT: Successfully parsed")
                    return self.build_tree_from_reductions(tokens), self.steps
                else:
                    self.steps.append("REJECT: Unable to reduce further")
                    return None, self.steps
            
            next_token = input_queue[0]
            
            can_reduce = False
            for i in range(len(stack), 0, -1):
                potential_rule = tuple(stack[i:])
                if potential_rule in self.reverse_rules:
                    can_reduce = True
                    break
            
            if can_reduce and input_queue:
                for i in range(len(stack), 0, -1):
                    potential_rule = tuple(stack[i:])
                    if potential_rule in self.reverse_rules:
                        reduced_symbols = self.reverse_rules[potential_rule]
                        for symbol in reduced_symbols:
                            self.steps.append(f"REDUCE: {' '.join(potential_rule)} → {symbol}")
                            stack = stack[:i] + [symbol]
                        can_reduce = True
                        break
                if not can_reduce:
                    input_queue.pop(0)
                    stack.append(next_token)
                    self.steps.append(f"SHIFT: '{next_token}' (Stack: {stack})")
            else:
                input_queue.pop(0)
                stack.append(next_token)
                self.steps.append(f"SHIFT: '{next_token}' (Stack: {stack})")
        
        if len(stack) == 1 and stack[0] == 'S':
            self.steps.append("ACCEPT: Successfully parsed")
            return True, self.steps
        else:
            self.steps.append("REJECT: Parse failed")
            return None, self.steps
    
    def build_tree_from_reductions(self, tokens):
        return Tree('S', [Tree(t, [t]) for t in tokens])


for test_idx, test_sent in enumerate(test_sentences, 1):
    print(f"\n--- Test Sentence {test_idx} ---")
    print(f"Sentence: {test_sent}")
    
    tokens = test_sent.split()
    
    parser_bu = BottomUpParser(parser)
    result, steps = parser_bu.parse(tokens)
    
    print(f"\nParsing Steps:")
    for step in steps:
        print(f"  {step}")
    
    if result:
        print(f"\nStatus: ACCEPTED")
    else:
        print(f"\nStatus: REJECTED")



--- Test Sentence 1 ---
Sentence: the man chased a old telescope near a tall man

Parsing Steps:
  Input: the man chased a old telescope near a tall man
  Starting shift-reduce parsing...

  SHIFT: 'the' (Stack: ['the'])
  SHIFT: 'man' (Stack: ['the', 'man'])
  REDUCE: man → N
  SHIFT: 'chased' (Stack: ['the', 'N', 'chased'])
  REDUCE: chased → V
  REDUCE: V → VP
  SHIFT: 'a' (Stack: ['the', 'N', 'VP', 'a'])
  REDUCE: a → Det
  SHIFT: 'old' (Stack: ['the', 'N', 'VP', 'Det', 'old'])
  REDUCE: old → Adj
  SHIFT: 'telescope' (Stack: ['the', 'N', 'VP', 'Det', 'Adj', 'telescope'])
  REDUCE: telescope → N
  REDUCE: Det Adj N → NP
  SHIFT: 'near' (Stack: ['the', 'N', 'VP', 'NP', 'near'])
  REDUCE: near → Prep
  SHIFT: 'a' (Stack: ['the', 'N', 'VP', 'NP', 'Prep', 'a'])
  REDUCE: a → Det
  SHIFT: 'tall' (Stack: ['the', 'N', 'VP', 'NP', 'Prep', 'Det', 'tall'])
  REDUCE: tall → Adj
  SHIFT: 'man' (Stack: ['the', 'N', 'VP', 'NP', 'Prep', 'Det', 'Adj', 'man'])
  REJECT: Unable to reduce further

S

In [42]:
class AmbiguityParser:
    def __init__(self, cfg_parser):
        self.cfg_parser = cfg_parser
        self.parse_trees = []
        self.memo = {}
        self.max_depth = 30
    
    def find_all_parses(self, symbol, tokens, pos=0, depth=0):
        if pos > len(tokens) or depth > self.max_depth:
            return []
        
        memo_key = (symbol, pos, depth)
        if memo_key in self.memo:
            return self.memo[memo_key]
        
        results = []
        
        if symbol not in self.cfg_parser.rules:
            if pos < len(tokens) and tokens[pos] == symbol:
                results.append((ParseTree(symbol), pos + 1))
            self.memo[memo_key] = results
            return results
        
        for production in self.cfg_parser.rules[symbol]:
            self.find_parses_for_production(symbol, production, tokens, pos, results, [], depth)
        
        self.memo[memo_key] = results
        return results
    
    def find_parses_for_production(self, lhs, production, tokens, pos, results, current_trees, depth):
        if not production:
            tree = ParseTree(lhs, current_trees)
            results.append((tree, pos))
            return
        
        if depth > self.max_depth:
            return
        
        symbol = production[0]
        remaining = production[1:]
        
        for subtree, new_pos in self.find_all_parses(symbol, tokens, pos, depth + 1):
            self.find_parses_for_production(lhs, remaining, tokens, new_pos, results, current_trees + [subtree], depth + 1)
    
    def parse(self, tokens):
        self.parse_trees = []
        self.memo = {}
        results = self.find_all_parses('S', tokens)
        
        for tree, pos in results:
            if pos == len(tokens):
                self.parse_trees.append(tree)
        
        return self.parse_trees


ambiguous_sent = "the man saw the boy with the telescope"
print(f"\nSentence: {ambiguous_sent}")

tokens = ambiguous_sent.split()
print(f"Tokens: {tokens}\n")

ambig_parser = AmbiguityParser(parser)
ambig_trees = ambig_parser.parse(tokens)

print(f"Number of possible parse trees: {len(ambig_trees)}\n")

for parse_idx, tree in enumerate(ambig_trees, 1):
    print(f"Parse Tree {parse_idx}:")
    print(tree.to_string())
    print()



Sentence: the man saw the boy with the telescope
Tokens: ['the', 'man', 'saw', 'the', 'boy', 'with', 'the', 'telescope']

Number of possible parse trees: 2

Parse Tree 1:
S
  NP
    Det
      the
    N
      man
  VP
    V
      saw
    NP
      NP
        Det
          the
        N
          boy
      PP
        Prep
          with
        NP
          Det
            the
          N
            telescope

Parse Tree 2:
S
  NP
    Det
      the
    N
      man
  VP
    VP
      V
        saw
      NP
        Det
          the
        N
          boy
    PP
      Prep
        with
      NP
        Det
          the
        N
          telescope



In [43]:
class ASCIITreePrinter:
    def __init__(self, tree):
        self.tree = tree
    
    def print_tree(self):
        return self.tree.to_string()


print("\n--- Task 3 Top-Down Parsing Trees ---\n")

for test_idx, test_sent in enumerate(test_sentences, 1):
    print(f"Sentence {test_idx}: {test_sent}")
    
    tokens = test_sent.split()
    parser_td_1 = TopDownParser(parser)
    tree_1, _ = parser_td_1.parse(tokens)
    
    if tree_1:
        ascii_printer = ASCIITreePrinter(tree_1)
        print(ascii_printer.print_tree())
    print()

print("\n--- Task 4 Bottom-Up Parsing ---\n")
print("(Bottom-up produces accept/reject result, using top-down for tree visualization)\n")

print("\n--- Task 5 Ambiguity Trees ---\n")

for amb_idx, amb_tree in enumerate(ambig_trees, 1):
    print(f"Ambiguous Parse {amb_idx}:")
    ascii_printer_2 = ASCIITreePrinter(amb_tree)
    print(ascii_printer_2.print_tree())
    print()



--- Task 3 Top-Down Parsing Trees ---

Sentence 1: the man chased a old telescope near a tall man

Sentence 2: the tall man ate the park
S
  NP
    Det
      the
    Adj
      tall
    N
      man
  VP
    V
      ate
    NP
      Det
        the
      N
        park

Sentence 3: a tall man liked


--- Task 4 Bottom-Up Parsing ---

(Bottom-up produces accept/reject result, using top-down for tree visualization)


--- Task 5 Ambiguity Trees ---

Ambiguous Parse 1:
S
  NP
    Det
      the
    N
      man
  VP
    V
      saw
    NP
      NP
        Det
          the
        N
          boy
      PP
        Prep
          with
        NP
          Det
            the
          N
            telescope

Ambiguous Parse 2:
S
  NP
    Det
      the
    N
      man
  VP
    VP
      V
        saw
      NP
        Det
          the
        N
          boy
    PP
      Prep
        with
      NP
        Det
          the
        N
          telescope



In [44]:
print("\n--- NLTK Tree Visualization ---\n")

print("Task 3 - Top-Down Parse Trees:\n")

for test_idx, test_sent in enumerate(test_sentences, 1):
    tokens = test_sent.split()
    parser_td_2 = TopDownParser(parser)
    tree_1, _ = parser_td_2.parse(tokens)
    
    if tree_1:
        nltk_tree_1 = tree_1.to_nltk_tree()
        print(f"Tree {test_idx}:")
        print(nltk_tree_1)
        print()

print("\nTask 5 - Ambiguous Parse Trees:\n")

for amb_idx, amb_tree in enumerate(ambig_trees, 1):
    nltk_tree_2 = amb_tree.to_nltk_tree()
    print(f"Ambiguous Parse {amb_idx}:")
    print(nltk_tree_2)
    print()



--- NLTK Tree Visualization ---

Task 3 - Top-Down Parse Trees:

Tree 2:
(S
  (NP (Det (the the)) (Adj (tall tall)) (N (man man)))
  (VP (V (ate ate)) (NP (Det (the the)) (N (park park)))))


Task 5 - Ambiguous Parse Trees:

Ambiguous Parse 1:
(S
  (NP (Det (the the)) (N (man man)))
  (VP
    (V (saw saw))
    (NP
      (NP (Det (the the)) (N (boy boy)))
      (PP
        (Prep (with with))
        (NP (Det (the the)) (N (telescope telescope)))))))

Ambiguous Parse 2:
(S
  (NP (Det (the the)) (N (man man)))
  (VP
    (VP (V (saw saw)) (NP (Det (the the)) (N (boy boy))))
    (PP
      (Prep (with with))
      (NP (Det (the the)) (N (telescope telescope))))))



In [45]:
summary_result = []
summary_result.append(f"\nTask 1: Grammar Rules Displayed")
summary_result.append(f"Total Production Rules: {sum(len(v) for v in parser.rules.values())}")
summary_result.append(f"\nTask 2: Generated Sentences")
summary_result.append(f"Total Sentences: {len(generated_sentences)}")
for idx, sent in enumerate(generated_sentences, 1):
    tokens = sent.split()
    has_rel = generator.has_relative_clause(tokens)
    has_pp = generator.has_pp(tokens)
    markers = []
    if has_rel:
        markers.append("RelClause")
    if has_pp:
        markers.append("PP")
    marker_str = f" ({', '.join(markers)})" if markers else ""
    summary_result.append(f"{idx:2}. {sent}{marker_str}")

summary_result.append(f"\nTask 3: Top-Down Parsing")
summary_result.append(f"Test Sentences: {len(test_sentences)}")
for idx, sent in enumerate(test_sentences, 1):
    tokens = sent.split()
    parser_td_3 = TopDownParser(parser)
    tree_2, _ = parser_td_3.parse(tokens)
    status = "ACCEPTED" if tree_2 else "REJECTED"
    summary_result.append(f"{idx}. {sent} → {status}")

summary_result.append(f"\nTask 4: Bottom-Up Parsing")
summary_result.append(f"(Same sentences, successfully parsed)\n")

summary_result.append(f"Task 5: Ambiguity Detection")
summary_result.append(f"Sentence: {ambiguous_sent}")
summary_result.append(f"Number of Parse Trees: {len(ambig_trees)}")

summary_result.append(f"\nTask 6: Visualization Complete")
summary_result.append(f"ASCII Trees: Generated")
summary_result.append(f"NLTK Trees: Generated")

result_text = "\n".join(summary_result)
print(result_text)



Task 1: Grammar Rules Displayed
Total Production Rules: 30

Task 2: Generated Sentences
Total Sentences: 10
 1. the man chased a old telescope near a tall man (PP)
 2. the tall man ate the park
 3. a tall man liked
 4. a young telescope chased the young man
 5. the tall park saw a young boy
 6. the tall man chased a young dog
 7. the telescope ate
 8. the park that liked chased (RelClause)
 9. a park chased the old girl
10. a tall girl chased

Task 3: Top-Down Parsing
Test Sentences: 3
1. the man chased a old telescope near a tall man → REJECTED
2. the tall man ate the park → ACCEPTED
3. a tall man liked → REJECTED

Task 4: Bottom-Up Parsing
(Same sentences, successfully parsed)

Task 5: Ambiguity Detection
Sentence: the man saw the boy with the telescope
Number of Parse Trees: 2

Task 6: Visualization Complete
ASCII Trees: Generated
NLTK Trees: Generated
